# API Ingestion Demo

## Flow

API JSON → Load response → Convert JSON to DataFrame → Validate required fields → Save raw JSON → Save staging table → Save clean table → Generate ingestion log

## 1. Import Required Libraries

In [1]:
import pandas as pd
import json
import uuid
import shutil
from pathlib import Path
from datetime import datetime, timezone

## 2. Define Project Paths

In [2]:
def find_project_root(current_path: Path) -> Path:
    """
    Find project root by walking upward until the data/ folder is found.
    This makes the notebook work even if it is executed from notebooks/data_team/.
    """
    current_path = current_path.resolve()

    for path in [current_path] + list(current_path.parents):
        if (path / "data").exists():
            return path

    raise FileNotFoundError(
        "Could not find project root. Please make sure a 'data/' folder exists in the project."
    )


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(CURRENT_DIR)

# Mentor expected sample name.
input_path = PROJECT_ROOT / "data" / "sample_inputs" / "sample_api_response.json"

# Fallback to your current file name.
fallback_input_path = PROJECT_ROOT / "data" / "sample_inputs" / "customer_api.json"

if not input_path.exists() and fallback_input_path.exists():
    input_path = fallback_input_path

raw_dir = PROJECT_ROOT / "data" / "raw" / "api"
staging_dir = PROJECT_ROOT / "data" / "staging" / "api"
clean_dir = PROJECT_ROOT / "data" / "clean" / "api"
log_dir = PROJECT_ROOT / "logs"

raw_dir.mkdir(parents=True, exist_ok=True)
staging_dir.mkdir(parents=True, exist_ok=True)
clean_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

raw_output_path = raw_dir / "sample_api_response.json"
staging_output_path = staging_dir / "api_staging.csv"
clean_output_path = clean_dir / "api_clean.csv"
log_output_path = log_dir / "api_ingestion_log.json"

print("Current dir:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Input path:", input_path)
print("Input exists:", input_path.exists())
print("Raw output:", raw_output_path)
print("Staging output:", staging_output_path)
print("Clean output:", clean_output_path)
print("Log output:", log_output_path)

Current dir: f:\data\new\quanskill\DataVision_Duy\week2\notebooks\data_team
Project root: F:\data\new\quanskill\DataVision_Duy\week2
Input path: F:\data\new\quanskill\DataVision_Duy\week2\data\sample_inputs\customer_api.json
Input exists: True
Raw output: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\api\sample_api_response.json
Staging output: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\api\api_staging.csv
Clean output: F:\data\new\quanskill\DataVision_Duy\week2\data\clean\api\api_clean.csv
Log output: F:\data\new\quanskill\DataVision_Duy\week2\logs\api_ingestion_log.json


## 3. Validate Input JSON File

In [3]:
if not input_path.exists():
    raise FileNotFoundError(
        f"Input JSON file not found. Expected one of: "
        f"{PROJECT_ROOT / 'data' / 'sample_inputs' / 'sample_api_response.json'} "
        f"or {PROJECT_ROOT / 'data' / 'sample_inputs' / 'customer_api.json'}"
    )

if input_path.stat().st_size == 0:
    raise ValueError(f"Input JSON file is empty: {input_path}")

print("Input JSON validation passed.")

Input JSON validation passed.


## 4. Start Ingestion Run

In [4]:
run_id = str(uuid.uuid4())
source_name = "customer_api"
source_type = "api"
owner = "Nguyen Minh Duy"

start_time = datetime.now(timezone.utc).isoformat()

print("Run ID:", run_id)
print("Start time:", start_time)

Run ID: b763bcb7-49fd-486d-bd61-7836eca4ba29
Start time: 2026-06-01T04:41:52.734872+00:00


## 5. Load API Response JSON

In [5]:
try:
    with open(input_path, "r", encoding="utf-8") as file:
        api_response = json.load(file)

    status = "success"
    error_message = None

    print("API response loaded successfully.")
    print("Response type:", type(api_response))

except Exception as error:
    status = "failed"
    error_message = str(error)
    raise

API response loaded successfully.
Response type: <class 'dict'>


## 6. Preview API Response Structure

In [6]:
if isinstance(api_response, dict):
    print("Top-level keys:", list(api_response.keys()))
elif isinstance(api_response, list):
    print("List length:", len(api_response))
else:
    print("Response preview:", api_response)

Top-level keys: ['customers']


## 7. Save Raw JSON to Raw Layer

In [7]:
# Raw layer should preserve the original API response.
shutil.copy2(input_path, raw_output_path)

print("Raw API JSON copied to:", raw_output_path)

Raw API JSON copied to: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\api\sample_api_response.json


## 8. Convert JSON to DataFrame

In [8]:
# This handles common API response shapes:
# 1. {"customers": [...]}
# 2. {"data": [...]}
# 3. [...]
# 4. single JSON object

if isinstance(api_response, dict) and "customers" in api_response:
    records = api_response["customers"]
elif isinstance(api_response, dict) and "data" in api_response:
    records = api_response["data"]
elif isinstance(api_response, list):
    records = api_response
else:
    records = [api_response]

df_staging = pd.json_normalize(records)

records_read = len(df_staging)

print("JSON converted to DataFrame successfully.")
print("Records read:", records_read)
print("Columns:", df_staging.columns.tolist())

df_staging.head()

JSON converted to DataFrame successfully.
Records read: 15
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'country', 'created_at']


,customer_id,first_name,last_name,email,phone,country,created_at
0,1001,John,Doe,john.doe@example.com,+1-202-555-0111,USA,2026-05-22T09:15:00Z
1,1002,Emma,Smith,emma.smith@example.com,+44-20-7946-0958,UK,2026-05-22T10:30:00Z
2,1003,Minh,Nguyen,minh.nguyen@example.com,+84-909-123-456,Vietnam,2026-05-23T08:00:00Z
3,1004,Sophia,Brown,sophia.brown@example.com,+1-310-555-0145,USA,2026-05-23T09:20:00Z
4,1005,Liam,Wilson,liam.wilson@example.com,+61-2-9374-4000,Australia,2026-05-23T11:45:00Z


## 9. Validate Required Fields

In [9]:
required_fields = ["customer_id", "email", "created_at"]

missing_required_fields = [
    field for field in required_fields
    if field not in df_staging.columns
]

if missing_required_fields:
    validation_status = "failed"
    error_message = f"Missing required fields: {missing_required_fields}"
else:
    validation_status = "passed"

print("Required fields:", required_fields)
print("Missing required fields:", missing_required_fields)
print("Validation status:", validation_status)

if missing_required_fields:
    raise ValueError(error_message)

Required fields: ['customer_id', 'email', 'created_at']
Missing required fields: []
Validation status: passed


## 10. Save Flattened Table to Staging

In [10]:
df_staging.to_csv(staging_output_path, index=False, encoding="utf-8")

print("Staging output saved to:", staging_output_path)

Staging output saved to: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\api\api_staging.csv


## 11. Check Duplicate Rows and Missing Values

In [11]:
duplicate_count = int(df_staging.duplicated().sum())
missing_values = df_staging.isna().sum()
total_missing_values = int(missing_values.sum())

print("Duplicate rows:", duplicate_count)

print("\nMissing values by column:")
print(missing_values)

print("\nTotal missing values:", total_missing_values)

Duplicate rows: 0

Missing values by column:
customer_id    0
first_name     0
last_name      0
email          0
phone          0
country        0
created_at     0
dtype: int64

Total missing values: 0


## 12. Create Clean Output

In [12]:
df_clean = df_staging.drop_duplicates().copy()

# Drop fully empty rows if any exist.
df_clean = df_clean.dropna(axis=0, how="all")

records_valid = len(df_clean)
records_invalid = records_read - records_valid

print("Records read:", records_read)
print("Records valid after cleaning:", records_valid)
print("Records invalid / removed:", records_invalid)

df_clean.head()

Records read: 15
Records valid after cleaning: 15
Records invalid / removed: 0


,customer_id,first_name,last_name,email,phone,country,created_at
0,1001,John,Doe,john.doe@example.com,+1-202-555-0111,USA,2026-05-22T09:15:00Z
1,1002,Emma,Smith,emma.smith@example.com,+44-20-7946-0958,UK,2026-05-22T10:30:00Z
2,1003,Minh,Nguyen,minh.nguyen@example.com,+84-909-123-456,Vietnam,2026-05-23T08:00:00Z
3,1004,Sophia,Brown,sophia.brown@example.com,+1-310-555-0145,USA,2026-05-23T09:20:00Z
4,1005,Liam,Wilson,liam.wilson@example.com,+61-2-9374-4000,Australia,2026-05-23T11:45:00Z


## 13. Save Cleaned Table to Clean Layer

In [13]:
df_clean.to_csv(clean_output_path, index=False, encoding="utf-8")

print("Clean output saved to:", clean_output_path)

Clean output saved to: F:\data\new\quanskill\DataVision_Duy\week2\data\clean\api\api_clean.csv


## 14. Generate Ingestion Log

In [14]:
end_time = datetime.now(timezone.utc).isoformat()

ingestion_log = {
    "run_id": run_id,
    "source_name": source_name,
    "source_type": source_type,
    "input_path_or_url": str(input_path),
    "start_time": start_time,
    "end_time": end_time,
    "status": status,
    "validation_status": validation_status,
    "records_read": int(records_read),
    "records_valid": int(records_valid),
    "records_invalid": int(records_invalid),
    "duplicate_rows_removed": int(duplicate_count),
    "missing_values": missing_values.astype(int).to_dict(),
    "total_missing_values": total_missing_values,
    "required_fields": required_fields,
    "missing_required_fields": missing_required_fields,
    "error_message": error_message,
    "raw_output_path": str(raw_output_path),
    "staging_output_path": str(staging_output_path),
    "clean_output_path": str(clean_output_path),
    "owner": owner
}

with open(log_output_path, "w", encoding="utf-8") as file:
    json.dump(ingestion_log, file, indent=4, ensure_ascii=False)

print("Ingestion log saved to:", log_output_path)
ingestion_log

Ingestion log saved to: F:\data\new\quanskill\DataVision_Duy\week2\logs\api_ingestion_log.json


{'run_id': 'b763bcb7-49fd-486d-bd61-7836eca4ba29',
 'source_name': 'customer_api',
 'source_type': 'api',
 'input_path_or_url': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\data\\sample_inputs\\customer_api.json',
 'start_time': '2026-06-01T04:41:52.734872+00:00',
 'end_time': '2026-06-01T04:42:15.291752+00:00',
 'status': 'success',
 'validation_status': 'passed',
 'records_read': 15,
 'records_valid': 15,
 'records_invalid': 0,
 'duplicate_rows_removed': 0,
 'missing_values': {'customer_id': 0,
  'first_name': 0,
  'last_name': 0,
  'email': 0,
  'phone': 0,
  'country': 0,
  'created_at': 0},
 'total_missing_values': 0,
 'required_fields': ['customer_id', 'email', 'created_at'],
 'missing_required_fields': [],
 'error_message': None,
 'raw_output_path': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\data\\raw\\api\\sample_api_response.json',
 'staging_output_path': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\data\\staging\\api\\api_staging.csv',
 'clean_output_path': '

## 15. Final Output Check

In [15]:
print("Expected outputs:")

print("Raw output exists:", raw_output_path.exists())
print("Staging output exists:", staging_output_path.exists())
print("Clean output exists:", clean_output_path.exists())
print("Log output exists:", log_output_path.exists())

print("\nOutput paths:")
print("Raw:", raw_output_path)
print("Staging:", staging_output_path)
print("Clean:", clean_output_path)
print("Log:", log_output_path)

Expected outputs:
Raw output exists: True
Staging output exists: True
Clean output exists: True
Log output exists: True

Output paths:
Raw: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\api\sample_api_response.json
Staging: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\api\api_staging.csv
Clean: F:\data\new\quanskill\DataVision_Duy\week2\data\clean\api\api_clean.csv
Log: F:\data\new\quanskill\DataVision_Duy\week2\logs\api_ingestion_log.json


## 16. Summary

```text
data/raw/api/sample_api_response.json
data/staging/api/api_staging.csv
data/clean/api/api_clean.csv
logs/api_ingestion_log.json
```